In [3]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.neighbors import KNeighborsRegressor  # needs scikit-learn


# ---------- CONFIG ----------
DATA_DIR = os.path.expanduser("~/Downloads/structure")  # change this to your data directory
PLOT_DIR = os.path.join(DATA_DIR, "plot")
C_SOUND = 343.0  # m/s
R_MM = 53.0
R_M = R_MM / 1000.0  # convert mm → meters
os.makedirs(PLOT_DIR, exist_ok=True)


def parse_dw_from_filename(fname):
    """
    Parse d and w (in mm) from filenames like 'd=10mm w=20mm.txt'.
    """
    base = os.path.basename(fname)
    # Example pattern: d=10mm w=20mm.txt
    pattern = r"d\s*=\s*([0-9]*\.?[0-9]+)\s*mm\s*w\s*=\s*([0-9]*\.?[0-9]+)\s*mm"
    m = re.search(pattern, base)
    if not m:
        raise ValueError(f"Cannot parse d and w from filename: {base}")
    d = float(m.group(1))
    w = float(m.group(2))
    return d, w


def load_band_data(filepath):
    """
    Load two-column band data (k, f) from txt, ignoring comment lines starting with '%'.
    Only keep points where k > 0.3.
    """
    data = np.loadtxt(filepath, comments='%')
    if data.ndim == 1:
        data = data.reshape(-1, 2)

    k = data[:, 0]
    f = data[:, 1]

    # Remove NaN frequencies
    mask = ~np.isnan(f)

    # Keep only k > 0.3
    mask &= (k > 0.3)

    return k[mask], f[mask]



def split_into_bands(k, f):
    """
    Split (k, f) into separate bands based on k resetting (k decreasing).
    Returns list of (k_band, f_band).
    """
    bands_k = []
    bands_f = []

    start_idx = 0
    for i in range(1, len(k)):
        if k[i] < k[i - 1]:  # new band starts
            bands_k.append(k[start_idx:i])
            bands_f.append(f[start_idx:i])
            start_idx = i
    # last band
    if start_idx < len(k):
        bands_k.append(k[start_idx:])
        bands_f.append(f[start_idx:])

    return list(zip(bands_k, bands_f))


def compute_total_band_gap(bands):
    """
    Given list of bands [(k_band, f_band), ...], compute total band gap.
    Band width = max(f) - min(f) (not used directly here).
    Band gap between band i and i+1 = min(f_{i+1}) - max(f_i) if positive.
    Total band gap = sum of all positive gaps.
    """
    # Sort bands by their central frequency (or min frequency)
    band_ranges = []
    for k_band, f_band in bands:
        f_min = np.min(f_band)
        f_max = np.max(f_band)
        band_ranges.append((f_min, f_max))

    band_ranges.sort(key=lambda x: x[0])

    total_gap = 0.0
    for i in range(len(band_ranges) - 1):
        fmin_next, fmax_next = band_ranges[i + 1]
        fmin_curr, fmax_curr = band_ranges[i]
        gap = fmin_next - fmax_curr
        if gap > 0:
            total_gap += gap

    return total_gap


def plot_band_diagram(bands, title, save_path):
    """
    Plot band diagram for one file.
    Each band is plotted with a different color (alpha=1),
    and the band region (between min and max frequency) is filled with transparency.
    """
    plt.figure(figsize=(6, 4))

    cmap = plt.cm.tab20
    num_bands = len(bands)

    for i, (k_band, f_band) in enumerate(bands):
        color = cmap(i % 20)

        # Plot the band line (solid, no transparency)
        plt.plot(
            k_band,
            f_band,
            '-',
            color=color,
            linewidth=1.5,
            alpha=1.0
        )

        # Compute band min/max
        f_min = np.min(f_band)
        f_max = np.max(f_band)

        # Fill the band region (higher transparency)
        plt.fill_between(
            k_band,
            f_min,
            f_max,
            color=color,
            alpha=0.25  # transparent band region
        )

    plt.xlabel(r"$k_x a_x$ (rad)")
    plt.ylabel("Frequency (Hz)")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()



def make_terrain_plot(df, save_path):
    """
    Make color terrain plot of total band gap vs w/R (x) and d/R (y),
    using KNN regression to interpolate/extrapolate.
    Uses a pure green colormap: white → light green → dark green.
    Also prints the (w/R, d/R) location of the maximum predicted bandgap.
    """
    x = df["w_over_R"].values  # w/R
    y = df["d_over_R"].values  # d/R
    z = df["total_band_gap"].values


    # Prepare grid
    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()

    x_pad = 0.05 * (x_max - x_min) if x_max > x_min else 0.1
    y_pad = 0.05 * (y_max - y_min) if y_max > y_min else 0.1

    x_grid = np.linspace(x_min - x_pad, x_max + x_pad, 200)
    y_grid = np.linspace(y_min - y_pad, y_max + y_pad, 200)
    Xg, Yg = np.meshgrid(x_grid, y_grid)

    # Train KNN regressor
    X_train = np.column_stack([x, y])
    knn = KNeighborsRegressor(n_neighbors=min(5, len(X_train)), weights="distance")
    knn.fit(X_train, z)

    Zg = knn.predict(np.column_stack([Xg.ravel(), Yg.ravel()])).reshape(Xg.shape)

    # ---- FIND MAXIMUM BANDGAP POINT (EXTRAPOLATED OR INTERPOLATED) ----
    idx = np.unravel_index(np.argmax(Zg), Zg.shape)
    max_w_over_R = Xg[idx]
    max_d_over_R = Yg[idx]
    max_gap = Zg[idx]

    print("\n=== Maximum Bandgap from Terrain (Interpolated/Extrapolated) ===")
    print(f"Max bandgap: {max_gap:.4f}")
    print(f"At w/R = {max_w_over_R:.4f}")
    print(f"At d/R = {max_d_over_R:.4f}")
    print("==============================================================\n")

    # --- MULTICOLOR COLORMAP FOR STRONG DIFFERENTIATION ---
    cmap_multi = plt.cm.turbo   # high-contrast scientific colormap



    # Plot
    plt.figure(figsize=(6, 5))
    cf = plt.pcolormesh(Xg, Yg, Zg, shading="auto", cmap=cmap_multi)
    cbar = plt.colorbar(cf)
    #cbar.set_label("Total band gap (Hz)")
    cbar.set_label("Normalized band gap (2π f R / C)")


    # Scatter original points
    #plt.scatter(x, y, c=z, cmap=cmap_multi, edgecolor="k", s=40)

    # Mark the maximum point
    plt.plot(max_w_over_R, max_d_over_R, 'o',color='hotpink', markersize=7)
    plt.plot(max_w_over_R, max_d_over_R, '.',color='lightgreen', markersize=7)
    plt.text(max_w_over_R, max_d_over_R,
             f"{max_gap:.4f}",
             color="red", fontsize=10,
             ha="right", va="top")

    plt.xlabel(r"$w/R$")
    plt.ylabel(r"$d/R$")
    plt.title("Total band gap map (color scale)")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


def main():
    records = []

    txt_files = glob.glob(os.path.join(DATA_DIR, "*.txt"))
    if not txt_files:
        print("No txt files found.")
        return

    for filepath in txt_files:
        try:
            d_mm, w_mm = parse_dw_from_filename(filepath)
        except ValueError as e:
            print(e)
            continue

        k, f = load_band_data(filepath)
        if len(k) == 0:
            print(f"No valid data in {filepath}")
            continue

        bands = split_into_bands(k, f)
        #total_gap = compute_total_band_gap(bands)
        raw_gap = compute_total_band_gap(bands)
        total_gap = 2 * np.pi * raw_gap * R_M / C_SOUND


        # Plot band diagram
        fname = os.path.basename(filepath)
        plot_path = os.path.join(PLOT_DIR, f"{os.path.splitext(fname)[0]}_band.png")
        plot_band_diagram(bands, title=fname, save_path=plot_path)

        # Store record
        d_over_R = d_mm / R_MM
        w_over_R = w_mm / R_MM
        records.append({
            "d_over_R": d_over_R,
            "w_over_R": w_over_R,
            "total_band_gap": total_gap
        })

    # Create DataFrame
    df = pd.DataFrame(records, columns=["d_over_R", "w_over_R", "total_band_gap"])
    print("\nData frame (d/R, w/R, total band gap):")
    print(df)

    # ---- NEW: Print maximum bandgap from actual data files ----
    if len(df) > 0:
        idx = df["total_band_gap"].idxmax()
        max_gap_data = df.loc[idx, "total_band_gap"]
        max_w_data = df.loc[idx, "w_over_R"]
        max_d_data = df.loc[idx, "d_over_R"]

        print("\n=== Maximum Bandgap from Actual Data Files ===")
        print(f"Max normalised bandgap: {max_gap_data:.4f}")
        print(f"At w/R = {max_w_data:.4f}")
        print(f"At d/R = {max_d_data:.4f}")
        print("================================================\n")

    # Terrain plot
    if len(df) >= 3:  # need at least a few points for interpolation
        terrain_path = os.path.join(PLOT_DIR, "bandgap_terrain_multicolor.png")
        make_terrain_plot(df, terrain_path)
        print(f"\nTerrain plot saved to: {terrain_path}")
    else:
        print("\nNot enough data points to build a meaningful terrain plot.")


if __name__ == "__main__":
    main()



Data frame (d/R, w/R, total band gap):
    d_over_R  w_over_R  total_band_gap
0   1.886792  0.943396        1.615052
1   2.452830  0.660377        1.648993
2   2.169811  0.943396        1.675107
3   2.358491  0.660377        1.660767
4   1.981132  0.849057        1.633756
..       ...       ...             ...
77  2.075472  0.943396        1.660220
78  2.452830  0.377358        1.580893
79  1.886792  0.849057        1.579129
80  2.358491  0.377358        1.601186
81  2.169811  0.849057        1.671004

[82 rows x 3 columns]

=== Maximum Bandgap from Actual Data Files ===
Max normalised bandgap: 1.6987
At w/R = 1.5094
At d/R = 2.1698


=== Maximum Bandgap from Terrain (Interpolated/Extrapolated) ===
Max bandgap: 1.6979
At w/R = 1.5114
At d/R = 2.1714


Terrain plot saved to: /Users/s.jack/Downloads/structure/plot/bandgap_terrain_multicolor.png


In [4]:
# ---------- CONFIG ----------
DATA_DIR = os.path.expanduser("~/Downloads/structure")  # change this to your data directory
PLOT_DIR = os.path.join(DATA_DIR, "plot")
C_SOUND = 343.0  # m/s
R_M = R_MM / 1000.0  # convert mm → meters
R_MM = 53.0
os.makedirs(PLOT_DIR, exist_ok=True)


def parse_dw_from_filename(fname):
    """
    Parse d and w (in mm) from filenames like 'd=10mm w=20mm.txt'.
    """
    base = os.path.basename(fname)
    # Example pattern: d=10mm w=20mm.txt
    pattern = r"d\s*=\s*([0-9]*\.?[0-9]+)\s*mm\s*w\s*=\s*([0-9]*\.?[0-9]+)\s*mm"
    m = re.search(pattern, base)
    if not m:
        raise ValueError(f"Cannot parse d and w from filename: {base}")
    d = float(m.group(1))
    w = float(m.group(2))
    return d, w


def load_band_data(filepath):
    """
    Load two-column band data (k, f) from txt, ignoring comment lines starting with '%'.
    Only keep points where k > 0.3.
    """
    data = np.loadtxt(filepath, comments='%')
    if data.ndim == 1:
        data = data.reshape(-1, 2)

    k = data[:, 0]
    f = data[:, 1]

    # Remove NaN frequencies
    mask = ~np.isnan(f)

    # Keep only k > 0.3
    mask &= (k > 0.3)

    return k[mask], f[mask]



def split_into_bands(k, f):
    """
    Split (k, f) into separate bands based on k resetting (k decreasing).
    Returns list of (k_band, f_band).
    """
    bands_k = []
    bands_f = []

    start_idx = 0
    for i in range(1, len(k)):
        if k[i] < k[i - 1]:  # new band starts
            bands_k.append(k[start_idx:i])
            bands_f.append(f[start_idx:i])
            start_idx = i
    # last band
    if start_idx < len(k):
        bands_k.append(k[start_idx:])
        bands_f.append(f[start_idx:])

    return list(zip(bands_k, bands_f))


def compute_total_band_gap(bands):
    """
    Given list of bands [(k_band, f_band), ...], compute total band gap.
    Band width = max(f) - min(f) (not used directly here).
    Band gap between band i and i+1 = min(f_{i+1}) - max(f_i) if positive.
    Total band gap = sum of all positive gaps.
    """
    # Sort bands by their central frequency (or min frequency)
    band_ranges = []
    for k_band, f_band in bands:
        f_min = np.min(f_band)
        f_max = np.max(f_band)
        band_ranges.append((f_min, f_max))

    band_ranges.sort(key=lambda x: x[0])

    total_gap = 0.0
    for i in range(len(band_ranges) - 1):
        fmin_next, fmax_next = band_ranges[i + 1]
        fmin_curr, fmax_curr = band_ranges[i]
        gap = fmin_next - fmax_curr
        if gap > 0:
            total_gap += gap

    return total_gap


def plot_band_diagram(bands, title, save_path):
    """
    Plot band diagram for one file.
    Each band is plotted with a different color (alpha=1),
    and the band region (between min and max frequency) is filled with transparency.
    """
    plt.figure(figsize=(6, 4))

    cmap = plt.cm.tab20
    num_bands = len(bands)

    for i, (k_band, f_band) in enumerate(bands):
        color = cmap(i % 20)

        # Plot the band line (solid, no transparency)
        plt.plot(
            k_band,
            f_band,
            '-',
            color=color,
            linewidth=1.5,
            alpha=1.0
        )

        # Compute band min/max
        f_min = np.min(f_band)
        f_max = np.max(f_band)

        # Fill the band region (higher transparency)
        plt.fill_between(
            k_band,
            f_min,
            f_max,
            color=color,
            alpha=0.25  # transparent band region
        )

    plt.xlabel(r"$k_x a_x$ (rad)")
    plt.ylabel("Frequency (Hz)")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()



def make_terrain_plot(df, save_path):
    """
    Make color terrain plot of total band gap vs w/R (x) and d/R (y),
    using KNN regression to interpolate/extrapolate.
    Uses a pure green colormap: white → light green → dark green.
    Also prints the (w/R, d/R) location of the maximum predicted bandgap.
    """
    x = df["w_over_R"].values  # w/R
    y = df["d_over_R"].values  # d/R
    z = df["total_band_gap"].values


    # Prepare grid
    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()

    x_pad = 0.05 * (x_max - x_min) if x_max > x_min else 0.1
    y_pad = 0.05 * (y_max - y_min) if y_max > y_min else 0.1

    x_grid = np.linspace(x_min - x_pad, x_max + x_pad, 200)
    y_grid = np.linspace(y_min - y_pad, y_max + y_pad, 200)
    Xg, Yg = np.meshgrid(x_grid, y_grid)

    # Train KNN regressor
    X_train = np.column_stack([x, y])
    knn = KNeighborsRegressor(n_neighbors=min(5, len(X_train)), weights="distance")
    knn.fit(X_train, z)

    Zg = knn.predict(np.column_stack([Xg.ravel(), Yg.ravel()])).reshape(Xg.shape)

    # ---- FIND MAXIMUM BANDGAP POINT (EXTRAPOLATED OR INTERPOLATED) ----
    idx = np.unravel_index(np.argmax(Zg), Zg.shape)
    max_w_over_R = Xg[idx]
    max_d_over_R = Yg[idx]
    max_gap = Zg[idx]

    print("\n=== Maximum Bandgap from Terrain (Interpolated/Extrapolated) ===")
    print(f"Max bandgap: {max_gap:.4f}")
    print(f"At w/R = {max_w_over_R:.4f}")
    print(f"At d/R = {max_d_over_R:.4f}")
    print("==============================================================\n")

    # --- PURE GREEN COLORMAP ---
    green_cmap = LinearSegmentedColormap.from_list(
        "green_ultra",
        [
            (1.0, 1.0, 1.0),     # white
            (0.95, 1.0, 0.95),   # very pale green
            (0.85, 1.0, 0.85),   # pale green
            (0.70, 1.0, 0.70),   # light green
            (0.50, 0.95, 0.50),  # medium green
            (0.30, 0.85, 0.30),  # strong green
            (0.15, 0.70, 0.15),  # darker green
            (0.05, 0.55, 0.05),  # deep green
            (0.00, 0.40, 0.00),  # very deep green
            (0.00, 0.25, 0.00)   # almost black-green (max bandgap)
        ]
    )


    # Plot
    plt.figure(figsize=(6, 5))
    cf = plt.pcolormesh(Xg, Yg, Zg, shading="auto", cmap=green_cmap)
    cbar = plt.colorbar(cf)
    #cbar.set_label("Total band gap (Hz)")
    cbar.set_label("Normalized band gap (2π f R / C)")


    # Scatter original points
    #plt.scatter(x, y, c=z, cmap=green_cmap, edgecolor="k", s=40)

    # Mark the maximum point
    plt.plot(max_w_over_R, max_d_over_R, 'o',color='hotpink', markersize=7)
    plt.plot(max_w_over_R, max_d_over_R, '.',color='lightgreen', markersize=7)
    plt.text(max_w_over_R, max_d_over_R,
             f"{max_gap:.4f}",
             color="red", fontsize=10,
             ha="left", va="bottom")

    plt.xlabel(r"$w/R$")
    plt.ylabel(r"$d/R$")
    plt.title("Total band gap map (green scale)")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


def main():
    records = []

    txt_files = glob.glob(os.path.join(DATA_DIR, "*.txt"))
    if not txt_files:
        print("No txt files found.")
        return

    for filepath in txt_files:
        try:
            d_mm, w_mm = parse_dw_from_filename(filepath)
        except ValueError as e:
            print(e)
            continue

        k, f = load_band_data(filepath)
        if len(k) == 0:
            print(f"No valid data in {filepath}")
            continue

        bands = split_into_bands(k, f)
        #total_gap = compute_total_band_gap(bands)
        raw_gap = compute_total_band_gap(bands)
        total_gap = 2 * np.pi * raw_gap * R_M / C_SOUND


        # Plot band diagram
        fname = os.path.basename(filepath)
        plot_path = os.path.join(PLOT_DIR, f"{os.path.splitext(fname)[0]}_band.png")
        plot_band_diagram(bands, title=fname, save_path=plot_path)

        # Store record
        d_over_R = d_mm / R_MM
        w_over_R = w_mm / R_MM
        records.append({
            "d_over_R": d_over_R,
            "w_over_R": w_over_R,
            "total_band_gap": total_gap
        })

    # Create DataFrame
    df = pd.DataFrame(records, columns=["d_over_R", "w_over_R", "total_band_gap"])
    print("\nData frame (d/R, w/R, total band gap):")
    print(df)

    # ---- NEW: Print maximum bandgap from actual data files ----
    if len(df) > 0:
        idx = df["total_band_gap"].idxmax()
        max_gap_data = df.loc[idx, "total_band_gap"]
        max_w_data = df.loc[idx, "w_over_R"]
        max_d_data = df.loc[idx, "d_over_R"]

        print("\n=== Maximum Bandgap from Actual Data Files ===")
        print(f"Max normalised bandgap: {max_gap_data:.4f}")
        print(f"At w/R = {max_w_data:.4f}")
        print(f"At d/R = {max_d_data:.4f}")
        print("================================================\n")

    # Terrain plot
    if len(df) >= 3:  # need at least a few points for interpolation
        terrain_path = os.path.join(PLOT_DIR, "bandgap_terrain_green.png")
        make_terrain_plot(df, terrain_path)
        print(f"\nTerrain plot saved to: {terrain_path}")
    else:
        print("\nNot enough data points to build a meaningful terrain plot.")


if __name__ == "__main__":
    main()


Data frame (d/R, w/R, total band gap):
    d_over_R  w_over_R  total_band_gap
0   1.886792  0.943396        1.615052
1   2.452830  0.660377        1.648993
2   2.169811  0.943396        1.675107
3   2.358491  0.660377        1.660767
4   1.981132  0.849057        1.633756
..       ...       ...             ...
77  2.075472  0.943396        1.660220
78  2.452830  0.377358        1.580893
79  1.886792  0.849057        1.579129
80  2.358491  0.377358        1.601186
81  2.169811  0.849057        1.671004

[82 rows x 3 columns]

=== Maximum Bandgap from Actual Data Files ===
Max normalised bandgap: 1.6987
At w/R = 1.5094
At d/R = 2.1698


=== Maximum Bandgap from Terrain (Interpolated/Extrapolated) ===
Max bandgap: 1.6979
At w/R = 1.5114
At d/R = 2.1714


Terrain plot saved to: /Users/s.jack/Downloads/structure/plot/bandgap_terrain_green.png


In [5]:
import random

d = random.randint(100, 130)
w = random.randint(60, 75)

print(f"d = {d}")
print(f"w = {w}")


d = 102
w = 62
